# Multi-Source Data Integration

AquaScope's unique strength is aggregating data from 10+ global sources into a unified schema.

This notebook demonstrates combining data from different sources for a comprehensive analysis.

In [ ]:
%matplotlib inline
import pandas as pd

## Available Data Sources

In [ ]:
sources = {
    "taiwan_moenv": "Taiwan EPA water quality (50+ stations, 10+ parameters)",
    "taiwan_wra_level": "Taiwan WRA river water levels",
    "taiwan_wra_reservoir": "Taiwan reservoir status (storage, inflow, outflow)",
    "usgs": "US Geological Survey real-time river data",
    "sdg6": "UN SDG 6 indicators (190+ countries)",
    "gemstat": "Global water quality monitoring (GEMStat/UNEP)",
    "taiwan_civil_iot": "Taiwan Civil IoT sensor network",
    "wqp": "US Water Quality Portal (EPA + USGS)",
    "openmeteo": "Open-Meteo global weather/forecast (free, no API key)",
    "copernicus": "Copernicus CDS / GloFAS global flood data",
}

for src, desc in sources.items():
    print(f"  {src:25s} — {desc}")

## Unified Schema

All collectors output standardised Pydantic objects:

- `WaterQualitySample` — parameter measurements with location
- `WaterLevelReading` — water level with timestamp
- `ReservoirStatus` — storage, inflow, outflow
- `SDG6Indicator` — country-level development metrics

In [ ]:
from aquascope.schemas.water_data import WaterQualitySample, DataSource

# Example: Create a sample manually
sample = WaterQualitySample(
    source=DataSource.TAIWAN_MOENV,
    station_id="TW001",
    station_name="Tamsui River Bridge",
    sample_datetime=pd.Timestamp("2024-06-15"),
    parameter="DO",
    value=7.8,
    unit="mg/L",
    river="Tamsui River",
    county="New Taipei",
)
print(sample.model_dump_json(indent=2))

## Quick Collection with Top-Level API

In [ ]:
# The simplest way to collect data
from aquascope import collect

# This is equivalent to:
#   collector = OpenMeteoCollector()
#   raw = collector.fetch_raw(lat=25.03, lon=121.57, ...)
#   records = collector.normalise(raw)
# But in one line:
# records = collect("openmeteo", lat=25.03, lon=121.57, mode="weather",
#                   start_date="2024-01-01", end_date="2024-01-07")

print("Use collect() for quick one-line data fetching")
print("Use individual collectors for fine-grained control")

## CLI Quick Reference

```bash
# Collect from any source
aquascope collect --source taiwan_moenv
aquascope collect --source usgs --days 7
aquascope collect --source openmeteo --lat 25.03 --lon 121.57 --mode weather

# Analyse
aquascope eda --file data/raw/water_data.json
aquascope quality --file data/raw/water_data.json --fix

# AI recommendations
aquascope recommend --from-file data/raw/water_data.json --goal "trend analysis"

# Natural language
aquascope solve "Forecast flooding at lat 25.03, lon 121.57"

# Hydrology
aquascope hydro --analysis fdc --file discharge.csv
aquascope hydro --analysis baseflow --file discharge.csv

# Visualise
aquascope plot --type timeseries --file data.csv --output plot.png
```